# **Dataset Maker**

Follow this guide to make your own custom dataset using default and previously tested parameters for all steps. 

**Discalimer:** It is recomended to have a copy of your data to start over since some of these steps are not revertible :O (even though no destructive editing to your data is applied).

## **1. Dataset Structure and Configuration**

### **Structure**

Before running any code you must make sure your data is organized as below. 

```
main/
├── raw/
│   ├── audio1.wav
│   ├── audio2.wav
│   └── ...
└── config/
    └── dataprep.json
```

### **Configuration**

You can create the configuration file `config/dataprep.json` by simple creating a `.txt` file, filling it with the text below, and then change its extension to `.json`.

```json
{
    "atoms_frames": 48,
    "atoms_hop_frames": 15,
    "crossfade_frames": 3
}
```

**Note:** You can eliminate train_split and val_split entries if you don't have premade splits.

## **2. Data Preparation**

From here, the data will be prepared in many steps. Make sure you write the path to your dataset in the following cell and then run the whole thing one by one. Some cells may take some time :)

### **2.1. Fix your dataset path**

In [ ]:
# Fill your dataset path here
path_to_dataset = "path/to/your/dataset"  # Change this to your dataset path <---------------------------

### **2.2. Precompute audio representation**

In [ ]:
from SCAPES.data.dataprep import atoms_maker

atoms_maker(path_to_dataset)

### **2.3. Initialize the dataset and split it**

By default, we will use 10 % of all data for training.

In [ ]:
from SCAPES.data.dataset import AtomSequenceDataset
from SCAPES.auxiliar.encodec_wrapper import EncodecProcessor
import torch

# Pick your device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize 48kHz processor
processor_48k_streamable = EncodecProcessor(sr=48000, streamable=True, device=device)

# 1. Default configuration
segment_length, context_length, hop_size = 5, 5, 1

# 2. Setup the main dataset with the new opt-in requested_keys
dataset = AtomSequenceDataset(
    dataset_path=path_to_dataset, 
    segment_length=segment_length,
    context_length=context_length,
    hop_size=hop_size,
    requested_keys=["latent_past", "latent_present", "latent_context_win", 
                    "scale_past", "scale_present", "scale_context_win", 
                    "index"],
    device=device)

dataset.make_split(val_split=0.1)

### **2.4. Precompute semantic annotations using CLAP**

In [ ]:
from SCAPES.data.dataprep import precompute_annotations
from SCAPES.models.factorization import load_global_encoder
from SCAPES.auxiliar.clap_wrapper import CLAPWrapper

method = "clap"
model = CLAPWrapper(version="2023", use_cuda=True)
precompute_annotations(
    dataset=dataset, 
    annotation_type=method, # "clap" or "custom"
    time_part="context_win",  # "past" or "context_win"
    model=model, 
    batch_size=128,
    device="cuda"
)

## **3. Visualize your annotations (optional)**

In [ ]:
from SCAPES.data.visualization import LatentSpaceExplorer
from SCAPES.data.dataset import AtomSequenceDataset

segment_length, context_length, hop_size = 5, 5, 1

# 2. Setup the main dataset with the new opt-in requested_keys
dataset = AtomSequenceDataset(
    dataset_path=path_to_dataset, 
    segment_length=segment_length,
    context_length=context_length,
    hop_size=hop_size,
    requested_keys=["latent_past", "latent_present", "latent_context_win", 
                    "scale_past", "scale_present", "scale_context_win", 
                    "clap_context_win", "index"],
    device="cpu")

# Initialize the visualizer (it will automatically detect the keys)
explorer = LatentSpaceExplorer(dataset=dataset, max_samples_per_file=300)

# Plot using PCA (Fast, shows global structure)
explorer.plot(method="pca")

# # Plot using t-SNE (Slower, but shows local clusters and "islands" beautifully)
explorer.plot(method="tsne")

## **4. Save your dataset**

You will need your preprocessed data for training. Run the following cells to zip your preprocessed data and then **make sure you download it**. 

Note: You can technically just leave it in your Google Drive folder and then train loading the data from there, however that is much slower.

In [ ]:
# Zip the full dataset directory (recursively) using Linux zip
zip_path = f"{path_to_dataset.rstrip('/')}_preprocessed.zip"
!zip -r "{zip_path}" "{path_to_dataset}"

print(f"✅ Created: {zip_path}")